In [6]:
from icecube import dataio, icetray, dataclasses
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
import sys
sys.path.insert(0, '/project/def-nahee/kbas/Graphnet-Applications')
from Metadata.paths import GCD

GCD_PATH = GCD['Spring2026MC']
print('GCD path:', GCD_PATH)

data_file   = dataio.I3File(GCD_PATH)
geometry    = data_file.pop_frame()
# calibration = data_file.pop_frame()

GCD path: /project/6008051/pone_simulation/GCD_Library/PONE_800mGrid_40mSpacing_40OMstring.i3.gz


In [8]:
print(type(geometry))
print('Stop:', geometry.Stop)
print('Keys:', list(geometry.keys()))

<class 'icecube._icetray.I3Frame'>
Stop: Geometry
Keys: ['StartTime', 'I3ModuleGeoMap', 'I3Geometry', 'Subdetectors', 'EndTime']


# I3OMGeoMap
OM-level positions (all PMTs in an OM share the same x,y,z)

In [9]:
om_map = geometry['I3Geometry'].omgeo
print('Number of OMs:', len(om_map))

sample_key, sample_geo = next(iter(om_map.items()))
attrs = [a for a in dir(sample_geo) if not a.startswith('_')]
skip  = {'position', 'orientation', 'omtype', 'area'}
attrs = [a for a in attrs if a not in skip]

rows = []
for omkey, omgeo in om_map.items():
    pos = omgeo.position
    ori = omgeo.orientation
    row = {
        'string': omkey.string,
        'om':     omkey.om,
        'pmt':    getattr(omkey, 'pmt', None),
        'x': pos.x, 'y': pos.y, 'z': pos.z,
        'dir_x': ori.dir.x, 'dir_y': ori.dir.y, 'dir_z': ori.dir.z,
        'omtype': int(omgeo.omtype),
        'area':   float(omgeo.area),
    }
    for name in attrs:
        v = getattr(omgeo, name)
        if isinstance(v, (int, float, str, bool)):
            row[name] = v
    rows.append(row)

om_df = pd.DataFrame(rows)
om_df = om_df.loc[:, om_df.nunique(dropna=False) > 1]
print(om_df.shape)
om_df.head(10)

Number of OMs: 903968
(903968, 9)


,string,om,pmt,x,y,z,dir_x,dir_y,dir_z
0,1,1,1,-139.9215,-762.1,-475.0,0.843391,0.000000e+00,5.372996e-01
1,1,1,2,-139.9215,-762.1,-475.0,0.843391,-5.372996e-01,6.123234e-17
2,1,1,3,-139.9215,-762.1,-475.0,0.843391,0.000000e+00,-5.372996e-01
3,1,1,4,-139.9215,-762.1,-475.0,0.843391,5.372996e-01,6.123234e-17
4,1,1,5,-139.9215,-762.1,-475.0,0.422618,6.408564e-01,6.408564e-01
5,1,1,6,-139.9215,-762.1,-475.0,0.422618,-6.408564e-01,6.408564e-01
6,1,1,7,-139.9215,-762.1,-475.0,0.422618,-6.408564e-01,-6.408564e-01
7,1,1,8,-139.9215,-762.1,-475.0,0.422618,6.408564e-01,-6.408564e-01
8,1,1,9,-140.0785,-762.1,-475.0,-0.843391,-1.032857e-16,5.372996e-01
9,1,1,10,-140.0785,-762.1,-475.0,-0.843391,5.372996e-01,-1.608123e-16


# I3ModuleGeoMap
Module-level positions (one row per module)

In [10]:
mod_map = geometry['I3ModuleGeoMap']
print('Number of modules:', len(mod_map))

k0, g0 = next(iter(mod_map.items()))
key_attrs = [a for a in dir(k0) if not a.startswith('_')]
geo_attrs = [a for a in dir(g0) if not a.startswith('_')]

rows = []
for mkey, mgeo in mod_map.items():
    pos = getattr(mgeo, 'pos', None) or getattr(mgeo, 'position', None)
    d   = getattr(mgeo, 'dir', None)
    row = {}
    for a in key_attrs:
        v = getattr(mkey, a)
        if isinstance(v, (int, float, str, bool)):
            row[f'key_{a}'] = v
    if pos:
        row['x'], row['y'], row['z'] = pos.x, pos.y, pos.z
    if d:
        row['dir_x'], row['dir_y'], row['dir_z'] = d.x, d.y, d.z
    for a in geo_attrs:
        if a in {'pos', 'position', 'orientation', 'dir'} or a in row:
            continue
        v = getattr(mgeo, a)
        if isinstance(v, (int, float, str, bool)):
            row[a] = v
    rows.append(row)

mod_df = pd.DataFrame(rows)
mod_df = mod_df.loc[:, mod_df.nunique(dropna=False) > 1]
print(mod_df.shape)
mod_df.head(10)

Number of modules: 56498
(56498, 5)


,key_om,key_string,x,y,z
0,1,1,-140.0,-762.1,-475.0
1,2,1,-140.0,-762.1,-450.0
2,3,1,-140.0,-762.1,-425.0
3,4,1,-140.0,-762.1,-400.0
4,5,1,-140.0,-762.1,-375.0
5,6,1,-140.0,-762.1,-350.0
6,7,1,-140.0,-762.1,-325.0
7,8,1,-140.0,-762.1,-300.0
8,9,1,-140.0,-762.1,-275.0
9,10,1,-140.0,-762.1,-250.0


# I3Geometry
Top-level geometry object summary

In [11]:
geo = geometry['I3Geometry']
public_attrs = [a for a in dir(geo) if not a.startswith('_')]

rows = []
for a in public_attrs:
    val = getattr(geo, a)
    n   = None
    try:
        n = len(val)
    except TypeError:
        pass
    rows.append({'attr': a, 'type': type(val).__name__, 'len': n, 'repr': repr(val)[:120]})

pd.DataFrame(rows)

,attr,type,len,repr
0,antennageo,I3AntennaGeoMap,0.0,<icecube._dataclasses.I3AntennaGeoMap object a...
1,end_time,I3Time,NaN,"I3Time(2039, 0L)"
2,from_frame,function,NaN,<Boost.Python.function object at 0x297a5e0>
3,iceactgeo,I3IceActGeoMap,0.0,<icecube._dataclasses.I3IceActGeoMap object at...
4,omgeo,I3OMGeoMap,903968.0,<icecube._dataclasses.I3OMGeoMap object at 0x1...
5,scintgeo,I3ScintGeoMap,0.0,<icecube._dataclasses.I3ScintGeoMap object at ...
6,snow_height_provenance,SnowHeightProvenance,NaN,icecube._dataclasses.SnowHeightProvenance.Unknown
7,start_time,I3Time,NaN,"I3Time(2019, 0L)"
8,stationgeo,I3StationGeoMap,0.0,<icecube._dataclasses.I3StationGeoMap object a...
9,tankgeo,method,NaN,<bound method tankgeo of <icecube._dataclasses...
